In [1]:
#!/usr/bin/env python
# -*- coding: utf-8 -*-

"""
Seizure Risk and Onset Prediction Model using TVB data
======================================================
This script uses The Virtual Brain (TVB) simulation data to train a machine learning
model that predicts both seizure risk (probability) and the time until seizure onset.
"""

import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from scipy.signal import find_peaks, welch
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import confusion_matrix, roc_curve, auc, mean_squared_error, r2_score
from sklearn.pipeline import Pipeline
import joblib
import os
from datetime import datetime
from tqdm import tqdm

# Import TVB modules
from tvb.simulator import simulator, models, coupling, integrators, monitors, noise
from tvb.datatypes import connectivity, surfaces, equations, patterns
from tvb.basic.neotraits.api import NArray

# Set random seed for reproducibility
np.random.seed(42)

# Create directory to store results
timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
results_dir = f'seizure_prediction_analysis_{timestamp}'
os.makedirs(results_dir, exist_ok=True)

# ====================================
# 1. Define custom classes for TVB
# ====================================

# Custom temporal equation for precise stimulus control
class PreciseStimulus(equations.TemporalApplicableEquation):
    onset = NArray(
        label="Onset time (ms)",
        default=np.array([0.0]),
        doc="Onset time of the stimulus")
    
    duration = NArray(
        label="Duration (ms)",
        default=np.array([0.0]),
        doc="Duration of the stimulus")
    
    amp = NArray(
        label="Amplitude",
        default=np.array([1.0]),
        doc="Amplitude of the stimulus")
    
    T = NArray(
        label="Period (ms)",
        default=np.array([1.0]),
        doc="Period between pulses")
    
    tau = NArray(
        label="Pulse width (ms)",
        default=np.array([0.5]),
        doc="Width of each pulse")

    def evaluate(self, t):
        t = np.asarray(t)
        mask = np.logical_and(t >= self.onset[0], t <= (self.onset[0] + self.duration[0]))
        result = np.zeros_like(t, dtype=float)
        t_rel = t[mask] - self.onset[0]
        pulse_mask = np.mod(t_rel, self.T[0]) <= self.tau[0]
        result[mask] = np.where(pulse_mask, self.amp[0], 0.0)
        return result

# Modified Epileptor model with enhanced stimulus response
class ModifiedEpileptorRestingState(models.EpileptorRestingState):
    def __init__(self, **kwargs):
        super().__init__(**kwargs)
        self.stim_history = 0.0
        
    def coupling_variables(self):
        return ["x1", "y1"]
    
    def update_state_variables(self, state_variables, coupling, stimulus):
        if stimulus is not None and np.any(stimulus != 0):
            # Accumulate stimulus effect with stronger weights
            self.stim_history = self.stim_history + 0.2 * stimulus
            
            # Update state variables with increased effect
            state_variables[0] = state_variables[0] + 1.2 * stimulus + 0.15 * self.stim_history  # x1
            state_variables[1] = state_variables[1] - 0.4 * stimulus  # y1 
            state_variables[2] = state_variables[2] + 0.35 * self.stim_history  # z
            
            # Modify system parameters with stronger effect
            self.r = self.r + 0.08 * stimulus
            self.x0 = self.x0 + 0.12 * self.stim_history
        
        # Slower decay for longer-lasting effects
        self.stim_history *= 0.97
        
        return super().update_state_variables(state_variables, coupling, stimulus)

# ====================================
# 2. Simulation and Feature Extraction Functions
# ====================================

def run_tvb_simulation(params, sim_length=5000.0, nodes_subset=None):
    """
    Run a TVB simulation with given parameters and extract time series data.
    
    Parameters:
    -----------
    params : dict
        Dictionary containing simulation parameters
    sim_length : float
        Length of simulation in milliseconds
    nodes_subset : list
        List of node indices to include in simulation output
        
    Returns:
    --------
    time : array
        Time points of simulation
    data : array
        Simulated brain activity data
    """
    # Initialize connectivity
    con = connectivity.Connectivity.from_file()
    nregions = len(con.region_labels)
    con.weights = con.weights - con.weights * np.eye(nregions)
    con.speed = np.array([params.get('speed', 3.0)])
    
    # Set up model
    epileptor_model = ModifiedEpileptorRestingState(
        a=np.array([params.get('a', 1.0)]),
        b=np.array([params.get('b', 3.0)]),
        c=np.array([params.get('c', 1.0)]),
        d=np.array([params.get('d', 5.0)]),
        r=np.array([params.get('r', 0.0)]),
        s=np.array([params.get('s', 4.0)]),
        x0=np.array([params.get('x0', -1.6)]),
        Iext=np.array([params.get('Iext', 3.1)])
    )
    
    # Set up coupling
    coup = coupling.Difference(a=np.array([params.get('coupling_strength', 0.1)]))
    
    # Set up stimulus
    stim_eqn = PreciseStimulus(
        onset=np.array([params.get('stim_onset', 1000.0)]),
        duration=np.array([params.get('stim_duration', 300.0)]),
        amp=np.array([params.get('stim_amp', 0.5)]),
        T=np.array([params.get('stim_period', 10.0)]),
        tau=np.array([params.get('stim_width', 5.0)])
    )
    
    # Define stimulus pattern (which regions receive stimulus)
    stim_pattern = np.zeros((nregions, 1))
    stim_regions = params.get('stim_regions', [0])  # Default to stimulating only first region
    for region in stim_regions:
        stim_pattern[region, 0] = 1.0
    
    stimulus = patterns.StimuliRegion(
        temporal=stim_eqn,
        connectivity=con,
        weight=stim_pattern
    )
    
    # Set up integrator and monitors
    integ = integrators.HeunDeterministic(dt=0.1)
    
    # Choose which monitors to use
    mon_type = params.get('monitor_type', 'temporalAverage')
    if mon_type == 'temporal':
        mon = monitors.Temporal(period=1.0)
    else:  # Default to temporal average
        mon = monitors.TemporalAverage(period=params.get('monitor_period', 10.0))
    
    # Add noise if specified
    if params.get('noise_sigma', 0.0) > 0:
        nsig = np.array([params.get('noise_sigma', 0.0)])
        hiss = noise.Additive(nsig=nsig)
        sim = simulator.Simulator(model=epileptor_model, connectivity=con, coupling=coup,
                                 integrator=integ, monitors=[mon], stimulus=stimulus, noise=hiss)
    else:
        sim = simulator.Simulator(model=epileptor_model, connectivity=con, coupling=coup,
                                 integrator=integ, monitors=[mon], stimulus=stimulus)
    
    # Configure simulation
    sim.configure()
    
    # Run simulation
    (time, data), = sim.run(simulation_length=sim_length)
    
    # If subset of nodes is requested, extract only those
    if nodes_subset is not None:
        data = data[:, nodes_subset]
    
    return time, data

def extract_features(time, data, window_size=100):
    """
    Extract features from simulation time series data.
    
    Parameters:
    -----------
    time : array
        Time points of simulation
    data : array
        Simulated brain activity data
    window_size : int
        Number of time points in each analysis window
        
    Returns:
    --------
    features : dict
        Dictionary of extracted features
    """
    features = {}
    n_regions = data.shape[1]
    
    # Global features across all brain regions
    all_region_data = data.reshape(len(time), -1)
    
    # Calculate mean amplitude across all regions
    features['global_mean_amplitude'] = np.mean(np.abs(all_region_data), axis=0)[0]
    
    # Calculate global variance
    features['global_variance'] = np.var(all_region_data)
    
    # Calculate global max amplitude
    features['global_max_amplitude'] = np.max(np.abs(all_region_data))
    
    # Calculate global min amplitude
    features['global_min_amplitude'] = np.min(np.abs(all_region_data))
    
    # Per-region features
    for i in range(n_regions):
        region_data = data[:, i, 0]  # Extract data for this region
        
        # Basic statistical features
        features[f'mean_r{i}'] = np.mean(region_data)
        features[f'std_r{i}'] = np.std(region_data)
        features[f'max_r{i}'] = np.max(region_data)
        features[f'min_r{i}'] = np.min(region_data)
        
        # Count signal crossings below seizure threshold
        threshold = -1.8  # Common threshold for seizure detection in Epileptor model
        crossings = np.sum(region_data < threshold)
        features[f'threshold_crossings_r{i}'] = crossings
        
        # Compute power spectral density
        if len(region_data) > 100:  # Only calculate if enough data points
            freqs, psd = welch(region_data, fs=1000.0/np.mean(np.diff(time)), nperseg=min(256, len(region_data)))
            
            # Extract frequency band powers
            delta_power = np.sum(psd[(freqs >= 0.5) & (freqs < 4)])
            theta_power = np.sum(psd[(freqs >= 4) & (freqs < 8)])
            alpha_power = np.sum(psd[(freqs >= 8) & (freqs < 13)])
            beta_power = np.sum(psd[(freqs >= 13) & (freqs < 30)])
            gamma_power = np.sum(psd[(freqs >= 30) & (freqs < 100)])
            
            features[f'delta_power_r{i}'] = delta_power
            features[f'theta_power_r{i}'] = theta_power
            features[f'alpha_power_r{i}'] = alpha_power
            features[f'beta_power_r{i}'] = beta_power
            features[f'gamma_power_r{i}'] = gamma_power
            
            # Compute power ratios (useful biomarkers)
            features[f'theta_alpha_ratio_r{i}'] = theta_power / (alpha_power + 1e-10)
            features[f'beta_alpha_ratio_r{i}'] = beta_power / (alpha_power + 1e-10)
            features[f'delta_theta_ratio_r{i}'] = delta_power / (theta_power + 1e-10)
        
        # Calculate peak characteristics
        peaks, properties = find_peaks(region_data, height=0, prominence=0.5)
        if len(peaks) > 0:
            features[f'peak_count_r{i}'] = len(peaks)
            features[f'mean_peak_height_r{i}'] = np.mean(properties['peak_heights'])
            features[f'max_peak_height_r{i}'] = np.max(properties['peak_heights'])
            
            # Calculate inter-peak intervals
            if len(peaks) > 1:
                intervals = np.diff(time[peaks])
                features[f'mean_peak_interval_r{i}'] = np.mean(intervals)
                features[f'std_peak_interval_r{i}'] = np.std(intervals)
        else:
            features[f'peak_count_r{i}'] = 0
            features[f'mean_peak_height_r{i}'] = 0
            features[f'max_peak_height_r{i}'] = 0
            features[f'mean_peak_interval_r{i}'] = 0
            features[f'std_peak_interval_r{i}'] = 0
    
    # Extract synchronization features between regions
    if n_regions > 1:
        # Calculate pairwise correlations
        corr_matrix = np.corrcoef(data[:, :, 0].T)
        features['mean_correlation'] = np.mean(corr_matrix[np.triu_indices(n_regions, k=1)])
        features['max_correlation'] = np.max(corr_matrix[np.triu_indices(n_regions, k=1)])
        features['min_correlation'] = np.min(corr_matrix[np.triu_indices(n_regions, k=1)])
        
        # Calculate phase synchronization (simplified approach)
        phase_sync = np.zeros((n_regions, n_regions))
        for i in range(n_regions):
            for j in range(i+1, n_regions):
                # Simple cross-correlation as a proxy for phase sync
                cross_corr = np.correlate(data[:, i, 0], data[:, j, 0], mode='same')
                phase_sync[i, j] = np.max(np.abs(cross_corr))
                phase_sync[j, i] = phase_sync[i, j]
        
        features['mean_phase_sync'] = np.mean(phase_sync[np.triu_indices(n_regions, k=1)])
        features['max_phase_sync'] = np.max(phase_sync[np.triu_indices(n_regions, k=1)])
    
    return features

def detect_seizure_onset(time, data, threshold=-1.8):
    """
    Detect the time of seizure onset in simulation data.
    
    Parameters:
    -----------
    time : array
        Time points of simulation
    data : array
        Simulated brain activity data
    threshold : float
        Threshold for seizure detection
        
    Returns:
    --------
    onset_time : float or None
        Time of seizure onset or None if no seizure detected
    """
    # Look for seizure in each region
    for i in range(data.shape[1]):
        region_data = data[:, i, 0]
        
        # Find where signal drops below threshold
        seizure_points = np.where(region_data < threshold)[0]
        
        # If seizure detected, return the first occurrence
        if len(seizure_points) > 0:
            return time[seizure_points[0]]
    
    # No seizure detected
    return None

def generate_training_data(n_samples=500, parameter_ranges=None):
    """
    Generate training data by running simulations with different parameters.
    
    Parameters:
    -----------
    n_samples : int
        Number of simulations to run
    parameter_ranges : dict
        Dictionary containing parameter ranges for simulation
        
    Returns:
    --------
    features_df : DataFrame
        DataFrame containing extracted features
    targets_df : DataFrame
        DataFrame containing seizure occurrence and onset time
    """
    # Default parameter ranges if none provided
    if parameter_ranges is None:
        parameter_ranges = {
            'a': (0.8, 1.2),
            'b': (2.5, 3.5),
            'c': (0.8, 1.2),
            'd': (4.5, 5.5),
            'r': (-0.2, 0.2),
            's': (3.8, 4.2),
            'x0': (-1.8, -1.4),
            'Iext': (2.8, 3.4),
            'coupling_strength': (0.0, 0.2),
            'stim_amp': (0.0, 1.0),
            'stim_duration': (200.0, 500.0),
            'stim_period': (5.0, 20.0),
            'stim_width': (2.0, 8.0),
            'noise_sigma': (0.0, 0.1)
        }
    
    # Storage for results
    all_features = []
    all_targets = {'seizure_occurred': [], 'onset_time': []}
    
    # Run simulations
    for i in tqdm(range(n_samples), desc="Generating simulations"):
        # Randomly sample parameters
        params = {}
        for param, (min_val, max_val) in parameter_ranges.items():
            params[param] = np.random.uniform(min_val, max_val)
        
        # Set fixed parameters
        params['stim_onset'] = 1000.0  # Start stimulus at 1000ms
        params['sim_length'] = 5000.0  # 5 seconds simulation
        
        # Run simulation
        try:
            time, data = run_tvb_simulation(params, sim_length=5000.0)
            
            # Extract features
            features = extract_features(time, data)
            
            # Add parameter values to features
            for param, value in params.items():
                features[f'param_{param}'] = value
            
            # Detect seizure
            onset_time = detect_seizure_onset(time, data)
            seizure_occurred = onset_time is not None
            
            # Calculate time to seizure onset (from stimulus onset)
            if seizure_occurred:
                time_to_onset = onset_time - params['stim_onset']
            else:
                time_to_onset = None
            
            # Store results
            all_features.append(features)
            all_targets['seizure_occurred'].append(seizure_occurred)
            all_targets['onset_time'].append(time_to_onset)
            
        except Exception as e:
            print(f"Simulation {i} failed: {e}")
            continue
    
    # Convert to DataFrames
    features_df = pd.DataFrame(all_features)
    targets_df = pd.DataFrame({
        'seizure_occurred': all_targets['seizure_occurred'],
        'onset_time': all_targets['onset_time']
    })
    
    return features_df, targets_df

# ====================================
# 3. Model Training and Evaluation
# ====================================

def train_seizure_prediction_models(features, targets, test_size=0.2, random_state=42):
    """
    Train models to predict seizure occurrence and onset time.
    
    Parameters:
    -----------
    features : DataFrame
        DataFrame containing extracted features
    targets : DataFrame
        DataFrame containing seizure occurrence and onset time
    test_size : float
        Fraction of data to use for testing
    random_state : int
        Random seed for reproducibility
        
    Returns:
    --------
    models : dict
        Dictionary containing trained models and scalers
    metrics : dict
        Dictionary containing model performance metrics
    """
    # Split data into training and testing sets
    X_train, X_test, y_train, y_test = train_test_split(
        features, targets, test_size=test_size, random_state=random_state
    )
    
    # Create classification model for seizure occurrence
    seizure_classifier = Pipeline([
        ('scaler', StandardScaler()),
        ('classifier', RandomForestClassifier(n_estimators=100, random_state=random_state))
    ])
    
    # Train classifier
    print("Training seizure occurrence classifier...")
    y_train_seizure = y_train['seizure_occurred']
    seizure_classifier.fit(X_train, y_train_seizure)
    
    # Evaluate classifier
    y_pred_seizure = seizure_classifier.predict(X_test)
    y_prob_seizure = seizure_classifier.predict_proba(X_test)[:,1]
    
    # Calculate metrics
    conf_matrix = confusion_matrix(y_test['seizure_occurred'], y_pred_seizure)
    fpr, tpr, _ = roc_curve(y_test['seizure_occurred'], y_prob_seizure)
    roc_auc = auc(fpr, tpr)
    
    # For onset time prediction, only use data where seizures occurred
    seizure_mask_train = y_train['seizure_occurred'] == True
    seizure_mask_test = y_test['seizure_occurred'] == True
    
    X_train_seizure = X_train[seizure_mask_train]
    y_train_onset = y_train.loc[seizure_mask_train, 'onset_time']
    
    X_test_seizure = X_test[seizure_mask_test]
    y_test_onset = y_test.loc[seizure_mask_test, 'onset_time']
    
    # Create regression model for onset time prediction
    if len(X_train_seizure) > 0:
        print("Training seizure onset time predictor...")
        onset_regressor = Pipeline([
            ('scaler', StandardScaler()),
            ('regressor', RandomForestRegressor(n_estimators=100, random_state=random_state))
        ])
        
        # Train regressor
        onset_regressor.fit(X_train_seizure, y_train_onset)
        
        # Evaluate regressor
        if len(X_test_seizure) > 0:
            y_pred_onset = onset_regressor.predict(X_test_seizure)
            mse = mean_squared_error(y_test_onset, y_pred_onset)
            r2 = r2_score(y_test_onset, y_pred_onset)
        else:
            mse = None
            r2 = None
    else:
        onset_regressor = None
        mse = None
        r2 = None
    
    # Store models and metrics
    models = {
        'seizure_classifier': seizure_classifier,
        'onset_regressor': onset_regressor
    }
    
    metrics = {
        'confusion_matrix': conf_matrix,
        'roc_auc': roc_auc,
        'fpr': fpr,
        'tpr': tpr,
        'mse': mse,
        'r2': r2
    }
    
    return models, metrics

def visualize_results(models, metrics, X_test, y_test, save_path):
    """
    Visualize model performance.
    
    Parameters:
    -----------
    models : dict
        Dictionary containing trained models
    metrics : dict
        Dictionary containing model performance metrics
    X_test : DataFrame
        Test features
    y_test : DataFrame
        Test targets
    save_path : str
        Path to save visualization
    """
    # Create figure with subplots
    fig, axs = plt.subplots(2, 2, figsize=(16, 14))
    
    # Plot ROC curve
    axs[0, 0].plot(metrics['fpr'], metrics['tpr'], label=f'AUC = {metrics["roc_auc"]:.3f}')
    axs[0, 0].plot([0, 1], [0, 1], 'k--')
    axs[0, 0].set_xlabel('False Positive Rate')
    axs[0, 0].set_ylabel('True Positive Rate')
    axs[0, 0].set_title('ROC Curve - Seizure Occurrence')
    axs[0, 0].legend(loc='lower right')
    
    # Plot confusion matrix
    conf_mat = metrics['confusion_matrix']
    axs[0, 1].imshow(conf_mat, cmap='Blues')
    axs[0, 1].set_title('Confusion Matrix - Seizure Occurrence')
    axs[0, 1].set_xlabel('Predicted')
    axs[0, 1].set_ylabel('Actual')
    
    # Add text annotations to confusion matrix
    axs[0, 1].text(0, 0, conf_mat[0, 0], ha='center', va='center')
    axs[0, 1].text(1, 0, conf_mat[0, 1], ha='center', va='center')
    axs[0, 1].text(0, 1, conf_mat[1, 0], ha='center', va='center')
    axs[0, 1].text(1, 1, conf_mat[1, 1], ha='center', va='center')
    axs[0, 1].set_xticks([0, 1])
    axs[0, 1].set_yticks([0, 1])
    axs[0, 1].set_xticklabels(['No Seizure', 'Seizure'])
    axs[0, 1].set_yticklabels(['No Seizure', 'Seizure'])
    
    # Plot feature importance for seizure classifier
    seizure_classifier = models['seizure_classifier'].named_steps['classifier']
    importances = seizure_classifier.feature_importances_
    feature_names = X_test.columns
    
    # Sort feature importances
    indices = np.argsort(importances)[-15:]  # Get the 15 most important features
    
    axs[1, 0].barh(range(len(indices)), importances[indices])
    axs[1, 0].set_yticks(range(len(indices)))
    axs[1, 0].set_yticklabels([feature_names[i] for i in indices])
    axs[1, 0].set_title('Feature Importance - Seizure Occurrence')
    axs[1, 0].set_xlabel('Importance')
    
    # Plot predicted vs actual onset times (if onset regressor exists)
    if models['onset_regressor'] is not None and metrics['mse'] is not None:
        seizure_mask = y_test['seizure_occurred'] == True
        X_test_seizure = X_test[seizure_mask]
        y_test_onset = y_test.loc[seizure_mask, 'onset_time']
        
        if len(X_test_seizure) > 0:
            y_pred_onset = models['onset_regressor'].predict(X_test_seizure)
            
            axs[1, 1].scatter(y_test_onset, y_pred_onset, alpha=0.5)
            
            # Add a perfect prediction line
            max_val = max(np.max(y_test_onset), np.max(y_pred_onset))
            min_val = min(np.min(y_test_onset), np.min(y_pred_onset))
            axs[1, 1].plot([min_val, max_val], [min_val, max_val], 'r--')
            
            axs[1, 1].set_xlabel('Actual Onset Time (ms)')
            axs[1, 1].set_ylabel('Predicted Onset Time (ms)')
            axs[1, 1].set_title(f'Seizure Onset Time Prediction\nMSE: {metrics["mse"]:.2f}, R²: {metrics["r2"]:.2f}')
        else:
            axs[1, 1].text(0.5, 0.5, 'No seizures in test data', ha='center', va='center')
            axs[1, 1].set_title('Seizure Onset Time Prediction')
    else:
        axs[1, 1].text(0.5, 0.5, 'No onset regressor trained', ha='center', va='center')
        axs[1, 1].set_title('Seizure Onset Time Prediction')
    
    plt.tight_layout()
    plt.savefig(os.path.join(save_path, 'model_performance.png'), dpi=300)
    plt.close()

def save_models(models, save_path):
    """
    Save trained models to disk.
    
    Parameters:
    -----------
    models : dict
        Dictionary containing trained models
    save_path : str
        Path to save models
    """
    # Save seizure classifier
    joblib.dump(models['seizure_classifier'], os.path.join(save_path, 'seizure_prediction_model.joblib'))
    
    # Save feature scaler separately for convenience
    joblib.dump(models['seizure_classifier'].named_steps['scaler'], 
                os.path.join(save_path, 'feature_scaler.joblib'))
    
    # Save onset regressor if it exists
    if models['onset_regressor'] is not None:
        joblib.dump(models['onset_regressor'], os.path.join(save_path, 'seizure_delay_model.joblib'))
        joblib.dump(models['onset_regressor'].named_steps['scaler'], 
                    os.path.join(save_path, 'delay_feature_scaler.joblib'))
    
    print(f"Models saved to {save_path}")

# ====================================
# 4. Main Pipeline
# ====================================

def main():
    """
    Main function to run the entire pipeline.
    """
    print("Starting seizure prediction model training pipeline...")
    
    # Generate training data (or load pre-existing data)
    existing_data = os.path.exists('features.csv') and os.path.exists('targets.csv')
    
    if existing_data:
        print("Loading existing data...")
        features = pd.read_csv('features.csv')
        targets = pd.read_csv('targets.csv')
    else:
        print("Generating training data from TVB simulations...")
        # Generate a smaller dataset for testing purposes
        # Increase n_samples for better model performance
        features, targets = generate_training_data(n_samples=100)
        
        # Save data for future use
        features.to_csv(os.path.join(results_dir, 'features.csv'), index=False)
        targets.to_csv(os.path.join(results_dir, 'targets.csv'), index=False)
    
    # Train models
    models, metrics = train_seizure_prediction_models(features, targets)
    
    # Split data again for visualization
    X_train, X_test, y_train, y_test = train_test_split(
        features, targets, test_size=0.2, random_state=42
    )
    
    # Visualize results
    visualize_results(models, metrics, X_test, y_test, results_dir)
    
    # Save models
    save_models(models, results_dir)
    
    print(f"Pipeline complete. Results saved to {results_dir}")
    
    # Print model performance summary
    print("\nModel Performance Summary:")
    print(f"Seizure Occurrence Prediction ROC AUC: {metrics['roc_auc']:.3f}")
    
    conf_mat = metrics['confusion_matrix']
    accuracy = (conf_mat[0, 0] + conf_mat[1, 1]) / np.sum(conf_mat)
    sensitivity = conf_mat[1, 1] / (conf_mat[1, 0] + conf_mat[1, 1]) if (conf_mat[1, 0] + conf_mat[1, 1]) > 0 else 0
    specificity = conf_mat[0, 0] / (conf_mat[0, 0] + conf_mat[0, 1]) if (conf_mat[0, 0] + conf_mat[0, 1]) > 0 else 0
    
    print(f"Accuracy: {accuracy:.3f}")
    print(f"Sensitivity: {sensitivity:.3f}")
    print(f"Specificity: {specificity:.3f}")
    
    if metrics['mse'] is not None:
        print(f"\nSeizure Onset Time Prediction:")
        print(f"Mean Squared Error: {metrics['mse']:.2f} ms²")
        print(f"R² Score: {metrics['r2']:.3f}")
    
    # Return models for potential further use
    return models, results_dir

# Run the main pipeline
if __name__ == "__main__":
    models, results_dir = main()
    
    # Example: using the model for prediction with new data
    print("\nExample: Predicting with new simulation parameters")
    
    # Create a parameter set that might lead to a seizure
    test_params = {
        'a': 1.0,
        'b': 3.0,
        'c': 1.0,
        'd': 5.0,
        'r': 0.0,
        's': 4.0,
        'x0': -1.7,  # Lower value increases seizure risk
        'Iext': 3.3,  # Higher external current increases seizure risk
        'coupling_strength': 0.15,
        'stim_amp': 0.8,
        'stim_onset': 1000.0,
        'stim_duration': 400.0,
        'stim_period': 10.0,
        'stim_width': 5.0,
        'noise_sigma': 0.05
    }
    
    # Run simulation
    time, data = run_tvb_simulation(test_params, sim_length=5000.0)
    
    # Extract features
    features = extract_features(time, data)
    
    # Add parameter values to features
    for param, value in test_params.items():
        features[f'param_{param}'] = value
    
    # Convert to DataFrame
    features_df = pd.DataFrame([features])
    
    # Fill any missing columns that the model might expect
    for col in models['seizure_classifier'].named_steps['scaler'].feature_names_in_:
        if col not in features_df.columns:
            features_df[col] = 0
    
    # Match columns and order to what the model expects
    features_df = features_df[models['seizure_classifier'].named_steps['scaler'].feature_names_in_]
    
    # Make predictions
    seizure_prob = models['seizure_classifier'].predict_proba(features_df)[0, 1]
    will_seize = models['seizure_classifier'].predict(features_df)[0]
    
    print(f"Seizure probability: {seizure_prob:.2%}")
    print(f"Seizure predicted: {'Yes' if will_seize else 'No'}")
    
    # If seizure is predicted and onset regressor exists, predict onset time
    if will_seize and models['onset_regressor'] is not None:
        # Make sure features match what the onset regressor expects
        onset_features = features_df[models['onset_regressor'].named_steps['scaler'].feature_names_in_]
        predicted_onset_time = models['onset_regressor'].predict(onset_features)[0]
        print(f"Predicted time until seizure onset: {predicted_onset_time:.1f} ms")
    
    # Visualize the simulation
    plt.figure(figsize=(12, 8))
    
    # Plot time series for first three regions
    n_regions = min(3, data.shape[1])
    for i in range(n_regions):
        plt.subplot(n_regions, 1, i+1)
        plt.plot(time, data[:, i, 0])
        plt.ylabel(f'Region {i}')
        plt.axhline(y=-1.8, color='r', linestyle='--', label='Seizure threshold')
        
        # Mark stimulus period
        stim_start = test_params['stim_onset']
        stim_end = stim_start + test_params['stim_duration']
        plt.axvspan(stim_start, stim_end, alpha=0.2, color='yellow', label='Stimulus')
        
        # Add legend for the first subplot only
        if i == 0:
            plt.legend()
    
    plt.xlabel('Time (ms)')
    plt.tight_layout()
    plt.savefig(os.path.join(results_dir, 'example_prediction.png'), dpi=300)
    
    print(f"Example visualization saved to {results_dir}/example_prediction.png")

/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/tvb/datatypes/surfaces.py:60: UserWarning: Geodesic distance module is unavailable; some functionality for surfaces will be unavailable.
  warnings.warn(msg)


Starting seizure prediction model training pipeline...
Generating training data from TVB simulations...


Generating simulations:   0%|          | 0/100 [00:00<?, ?it/s]

2025-03-02 09:57:28,566 - WARNING - tvb.basic.readers - File 'hemispheres' not found in ZIP.
Simulation 0 failed: Valid kwargs for type <class 'tvb.simulator.simulator.Simulator'> are: ('connectivity', 'conduction_speed', 'coupling', 'surface', 'stimulus', 'model', 'integrator', 'initial_conditions', 'monitors', 'simulation_length', 'gid'). You have given: 'noise'
2025-03-02 09:57:28,571 - WARNING - tvb.basic.readers - File 'hemispheres' not found in ZIP.
Simulation 1 failed: Valid kwargs for type <class 'tvb.simulator.simulator.Simulator'> are: ('connectivity', 'conduction_speed', 'coupling', 'surface', 'stimulus', 'model', 'integrator', 'initial_conditions', 'monitors', 'simulation_length', 'gid'). You have given: 'noise'
2025-03-02 09:57:28,575 - WARNING - tvb.basic.readers - File 'hemispheres' not found in ZIP.
Simulation 2 failed: Valid kwargs for type <class 'tvb.simulator.simulator.Simulator'> are: ('connectivity', 'conduction_speed', 'coupling', 'surface', 'stimulus', 'model', 

Generating simulations:  22%|██▏       | 22/100 [00:00<00:00, 212.48it/s]

...

Generating simulations:  89%|████████▉ | 89/100 [00:00<00:00, 215.66it/s]

Simulation 88 failed: Valid kwargs for type <class 'tvb.simulator.simulator.Simulator'> are: ('connectivity', 'conduction_speed', 'coupling', 'surface', 'stimulus', 'model', 'integrator', 'initial_conditions', 'monitors', 'simulation_length', 'gid'). You have given: 'noise'
2025-03-02 09:57:28,981 - WARNING - tvb.basic.readers - File 'hemispheres' not found in ZIP.
Simulation 89 failed: Valid kwargs for type <class 'tvb.simulator.simulator.Simulator'> are: ('connectivity', 'conduction_speed', 'coupling', 'surface', 'stimulus', 'model', 'integrator', 'initial_conditions', 'monitors', 'simulation_length', 'gid'). You have given: 'noise'
2025-03-02 09:57:28,985 - WARNING - tvb.basic.readers - File 'hemispheres' not found in ZIP.
Simulation 90 failed: Valid kwargs for type <class 'tvb.simulator.simulator.Simulator'> are: ('connectivity', 'conduction_speed', 'coupling', 'surface', 'stimulus', 'model', 'integrator', 'initial_conditions', 'monitors', 'simulation_length', 'gid'). You have give

Generating simulations: 100%|██████████| 100/100 [00:00<00:00, 212.01it/s]

Simulation 99 failed: Valid kwargs for type <class 'tvb.simulator.simulator.Simulator'> are: ('connectivity', 'conduction_speed', 'coupling', 'surface', 'stimulus', 'model', 'integrator', 'initial_conditions', 'monitors', 'simulation_length', 'gid'). You have given: 'noise'


ValueError: With n_samples=0, test_size=0.2 and train_size=None, the resulting train set will be empty. Adjust any of the aforementioned parameters.